# Алгоритм формирования категорий из товарных слов

In [ ]:
import h3
import os
import sys
import ast
import csv
import pickle
import numpy as np
import pandas as pd
from scipy import sparse
from tqdm import tqdm
from datetime import datetime
pd.set_option('display.max_columns', None)

os.environ['SPARK_MAJOR_VERSION'] = '3'
os.environ['SPARK_HOME'] = '/usr/sdp/current/spark3-client/'
os.environ['PYSPARK_DRIVER_PYTHON'] = 'python'
os.environ['LD_LIBRARY_PATH'] = '/opt/python/virtualenv/jupyter/lib'
os.environ['PYSPARK_PYTHON'] = '/data/sdp/mlpy3811v23/bin/python' #/opt/cloudera/parcels/PYENV.AUTOML/bin/python
 
sys.path.insert(0, '/usr/sdp/current/spark3-client/python/')
sys.path.insert(0, '/usr/sdp/current/spark3-client/python/lib/py4j_current')

from tqdm import tqdm
tqdm.pandas()
# from paths import *
import numpy as np
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.options.display.float_format = '{:.3f}'.format

import pyspark.sql.functions as F
import pyspark.sql.types as st
from pyspark.sql.functions import udf
from pyspark.sql.window import Window as W
from pyspark.sql.functions import row_number
from pyspark import SparkContext, SparkConf, HiveContext
from pyspark.sql.types import IntegerType, StringType, DecimalType, StructType, StructField, ArrayType, DoubleType
from pyspark.sql import SparkSession

conf = SparkConf().setAppName('Nikita wordstov Weights')\
    .setMaster("yarn")\
    .set('spark.executor.instances', '7')\
    .set('spark.executor.cores', '7')\
    .set('spark.executor.memory', '15g')\
    .set("spark.sql.parquet.int96RebaseModeInRead", "CORRECTED")\
    .set("spark.sql.parquet.int96RebaseMode", "CORRECTED")
    # .set('spark.driver.memory', '20g')\
    # .set('spark.driver.maxResultSize', '20g')\
print('Start', datetime.now())
sqlContext = SparkSession\
                .builder\
                .config(conf=conf)\
                .getOrCreate()
print('Allocated', datetime.now())


# Отбираем ИНН (берем для примера только 10 тысяч)

In [ ]:
# с этой даты до текущей рассматриваем все транзакции с товарными словами
# Если точнее, то с этой даты до 2025-05-30, т.к. с мая витрина товарных слов не обновлялась
START_HIST_WORDS = "2025-03-01" 

In [ ]:
# отбираем ИНН по предыдущим фильтрам в спарке
tmb_basis_client = sqlContext.table("arnsdpsbx_t_team_apm.tmb_basis_client")

In [ ]:
# ищем последнюю итерацию расчетов
tmb_basis_client.groupby("report_dt").agg(F.count(F.col("inn"))).orderBy("report_dt", ascending=False).show()

In [ ]:
inn_base_info = (
    tmb_basis_client
        .filter(F.col("report_dt")=="2025-08-31") # тут ставим последнюю дату, полученную из предыдущего расчета
        .filter(F.col("org_segment_sber").isin(["Микро", "Малые"]))
        .filter(F.col('active_flg') == 0)
        .filter(F.col("type").isin(["ИП", "ЮЛ"]))
).limit(10_000)

inn_base_info.persist()

In [ ]:
inn_base_info.count()

In [ ]:
all_need_inns = inn_base_info.select("inn")

## Выгружаем товарные слова

In [ ]:
prm_vitr_base = sqlContext.table("prx_smd_recsys_products_custom_cib_ml360_clients_products_extract.transactions_products_extract")

In [ ]:
prm_vitr = (
    prm_vitr_base.filter(F.col("short_dt")>=START_HIST_WORDS)
    .filter(F.col("inn_dt").isNotNull())
    .filter(F.col("inn_kt").isNotNull())
    .filter(F.col("inn_kt")!=F.col("inn_dt"))
    .drop("c_nazn", "c_kl_dt_2_kpp", "c_kl_kt_2_kpp", "okved_kt", "group_id", "word_v2_id", "group")
)

In [ ]:
mmb_inns_sample = prm_vitr.join(
    inn_base_info.select("inn"),
    prm_vitr_base.inn_kt==inn_base_info.inn, how="inner"
)

In [ ]:
mmb_inns_sample_fullstat = mmb_inns_sample.groupby("inn_kt").agg(
    F.countDistinct(F.col("id")).alias("all_nunique_cnt"),
    F.sum(F.col("c_sum") * F.col("sum_koef")).alias("all_sum"),
)

mmb_inns_sample_grpb = (
    mmb_inns_sample
        .filter(F.col("word").isNotNull())
        .groupby("inn_kt", "word").agg(
            F.sum(F.col("c_sum") * F.col("sum_koef")).alias("word_sum"),
            F.countDistinct(F.col("id")).alias("word_nunique_cnt"))
        .join(mmb_inns_sample_fullstat, how="left", on="inn_kt")
)

In [ ]:
# Промежуточные сохранения для ускорения повторных расчетов
mmb_inns_sample_grpb_df = mmb_inns_sample_grpb.toPandas()
mmb_inns_sample_grpb_df.to_parquet("./data/mmb_inns_sample_grpb_df__01_10.parquet", index=False)

In [ ]:
mmb_inns_sample_grpb_df = pd.read_parquet("./data/mmb_inns_sample_grpb_df__01_10.parquet")
mmb_inns_sample_grpb_df["word_sum"] = mmb_inns_sample_grpb_df["word_sum"].astype("float")
mmb_inns_sample_grpb_df["all_sum"] = mmb_inns_sample_grpb_df["all_sum"].astype("float")

In [ ]:
mmb_inns_sample_grpb_df["inn_kt"].nunique()

# Взвешиваем товарные слова для каждой компании

In [ ]:
def weight_by_scoring_vectorized(df):
    """
    Векторизованная версия взвешенного скоринга
    """
    df_weighted = df.copy()
    
    # Нормализация показателей в рамках каждой компании с помощью groupby + transform
    df_weighted['norm_sum'] = df_weighted.groupby('inn_kt')['word_sum'].transform(
        lambda x: x / x.max()
    ).astype("float64")
    
    df_weighted['norm_freq'] = df_weighted.groupby('inn_kt')['word_nunique_cnt'].transform(
        lambda x: x / x.max()
    ).astype("float64")
    
    # Заполняем NaN значения (если max = 0)
    df_weighted['norm_sum'] = df_weighted['norm_sum'].fillna(0)
    df_weighted['norm_freq'] = df_weighted['norm_freq'].fillna(0)

    # Вычисляем общий score
    df_weighted['total_score'] = 0.7 * df_weighted['norm_sum'] + 0.3 * df_weighted['norm_freq']
    
    # # Категоризируем на основе score
    # df_weighted['weight_category'] = df_weighted.groupby('inn_kt')['total_score'].transform(
    #     lambda x: pd.qcut(x, q=3, labels=['неважные', 'средние', 'важные'], duplicates='drop')
    # )
        
    return df_weighted

In [ ]:
# # Применяем векторизованный алгоритм
df_result_3_vec = weight_by_scoring_vectorized(mmb_inns_sample_grpb_df)

In [ ]:
df_result_3_vec.shape

In [ ]:
df_result_3_vec.head(3)

## Объединяем товарные слова с рубриками allbiz

In [ ]:
tovslova_map = pd.read_csv("./data/tovslova-allbz-WEIG__30-07__FULL.csv")
print(tovslova_map["Товарное слово в проме"].nunique(), tovslova_map[tovslova_map["allbiz_weights"]<3]["Товарное слово в проме"].nunique())

In [ ]:
tovslova_map = tovslova_map[tovslova_map["allbiz_weights"]<3]

In [ ]:
df_result_3_vec = df_result_3_vec.merge(
    tovslova_map[["Товарное слово в проме", "Категории_allbiz_step_3_CAT1", "Категории_allbiz_step_3_CAT2"]].rename(columns={"Товарное слово в проме": "word"}),
    how="left", on="word"
)

In [ ]:
df_result_3_vec_FIN = df_result_3_vec[["inn_kt", "word", "word_sum", "Категории_allbiz_step_3_CAT1", "Категории_allbiz_step_3_CAT2", "total_score"]].dropna()

In [ ]:
# df_result_3_vec_FIN[df_result_3_vec_FIN["inn_kt"]==""].sort_values(by="total_score", ascending=False)

Отдельный блок для статистики по топ-словам и для отбора конкурентов:

In [ ]:
# отдельно для подсчета статистики по словам:
df_result_3_vec_FIN_topwords = df_result_3_vec_FIN[["inn_kt", "word", "total_score"]].drop_duplicates()
df_result_3_vec_FIN_topwords["word_rank"] = df_result_3_vec_FIN_topwords.groupby("inn_kt")["total_score"].rank(ascending=False, method="dense")
df_result_3_vec_FIN_topwords = df_result_3_vec_FIN_topwords[df_result_3_vec_FIN_topwords["word_rank"]<=5]

Продолжаем вычисления:

In [ ]:
# суммируем все значения маппингов по словам
df_result_3_vec_FIN = df_result_3_vec_FIN.groupby(
    ["inn_kt", "Категории_allbiz_step_3_CAT1", "Категории_allbiz_step_3_CAT2"]).agg({"total_score": "sum", "word_sum": "sum"}).reset_index()
# Ранжируем все allbiz-категории
df_result_3_vec_FIN["total_rank"] = df_result_3_vec_FIN.groupby(
    ["inn_kt"])["total_score"].rank(ascending=False, method="dense")

In [ ]:
# df_result_3_vec_FIN[df_result_3_vec_FIN["inn_kt"]==""].sort_values(by="total_score", ascending=False).head(10)

В целом, на этом этапе получаем интересующий нас справочник вида `инн-категория_allbiz`. Можно теперь отбирать топ-категории по полю `total_rank`. Естественно, этот ранг формируется корректно, если есть записи в истории транзакций. Иначе - категории не формируются, т.к. нет исходных товарных слов.